In [1]:
import time
import torch
import numpy as np
import gymnasium as gym

from modules.agent_nets.modular import ModularAgentNetwork
from modules.backbones.mlp import MLPBackbone
from modules.heads.policy import PolicyHead
from modules.heads.value import ValueHead
from agents.learner.losses.representations import (
    ClassificationRepresentation,
    ScalarRepresentation,
)
from agents.action_selectors.selectors import CategoricalSelector
from agents.action_selectors.decorators import PPODecorator
from agents.action_selectors.policy_sources import NetworkPolicySource


from replay_buffers.modular_buffer import BufferConfig, ModularReplayBuffer
from replay_buffers.concurrency import LocalBackend
from replay_buffers.processors import (
    InputProcessor,
    GAEProcessor,
    StackedInputProcessor,
    LegalMovesMaskProcessor,
    AdvantageNormalizer,
)
from replay_buffers.writers import PPOWriter
from replay_buffers.samplers import WholeBufferSampler


from agents.learner.batch_iterators import PPOEpochIterator
from agents.learner.base import UniversalLearner
from agents.learner.losses.loss_pipeline import LossPipeline
from agents.learner.losses.policy import ClippedSurrogateLoss
from agents.learner.losses.value import ClippedValueLoss
from agents.learner.callbacks import MetricEarlyStopCallback
from agents.learner.target_builders import (
    PassThroughTargetBuilder,
    TargetBuilderPipeline,
    SingleStepFormatter,
)

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# --- Explicit Hyperparameters ---
ENV_ID = "CartPole-v1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# PPO Core Params
STEPS_PER_EPOCH = 512
NUM_MINIBATCHES = 4
TRAIN_POLICY_ITERATIONS = 4

# Algorithm Params
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_PARAM = 0.2
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
LEARNING_RATE = 2.5e-4
TARGET_KL = 0.02
TOTAL_STEPS = 256000

# --- Setup Environment & Dimensions ---
env = gym.make(ENV_ID)
obs_dim = env.observation_space.shape
num_actions = env.action_space.n

print(f"Environment: {ENV_ID} | Obs Dim: {obs_dim} | Num Actions: {num_actions}")

Using device: cpu
Environment: CartPole-v1 | Obs Dim: (4,) | Num Actions: 2


In [ ]:
# --- Instantiate Components Explicitly ---

# 1. Agent Network
# TODO: ensure proper initialization of orthogonal with sqrt 2 and bias 0, and output layers scale 0.01 for policy and value scale 1.0
agent_network = ModularAgentNetwork(
    input_shape=obs_dim,
    num_actions=num_actions,
    components={
        "policy_head": PolicyHead(
            input_shape=obs_dim,
            representation=ClassificationRepresentation(
                num_classes=num_actions,
            ),
            neck=MLPBackbone(
                input_shape=obs_dim,
                widths=[64, 64],
            ),
            noisy_sigma=0.0,
        ),
        "value_head": ValueHead(
            input_shape=obs_dim,
            representation=ScalarRepresentation(),
            neck=MLPBackbone(
                input_shape=obs_dim,
                widths=[64, 64],
            ),
            noisy_sigma=0.0,
        ),
    },
    minibatch_size=STEPS_PER_EPOCH // NUM_MINIBATCHES,
    # Pass your refactored arch kwargs here instead of a config object
    # e.g., hidden_dims=[64, 64]
).to(DEVICE)

# 2. Action Selector & Policy Source
action_selector = CategoricalSelector(exploration=True)
action_selector = PPODecorator(inner_selector=action_selector)
policy_source = NetworkPolicySource(agent_network)

# 3. Replay Buffer
configs = [
    BufferConfig("observations", shape=obs_dim, dtype=torch.float32),
    BufferConfig("actions", shape=(), dtype=torch.int64),
    BufferConfig("rewards", shape=(), dtype=torch.float32),
    BufferConfig("values", shape=(), dtype=torch.float32),
    BufferConfig("old_log_probs", shape=(), dtype=torch.float32),
    BufferConfig("legal_moves_masks", shape=(num_actions,), dtype=torch.bool),
    BufferConfig("advantages", shape=(), dtype=torch.float32),
    BufferConfig("returns", shape=(), dtype=torch.float32),
]

input_stack = StackedInputProcessor(
    [
        GAEProcessor(GAMMA, GAE_LAMBDA),
        LegalMovesMaskProcessor(
            num_actions, input_key="legal_moves", output_key="legal_moves_masks"
        ),
    ]
)

replay_buffer = ModularReplayBuffer(
    max_size=STEPS_PER_EPOCH,
    batch_size=STEPS_PER_EPOCH,
    buffer_configs=configs,
    input_processor=input_stack,
    output_processor=AdvantageNormalizer(),
    writer=PPOWriter(STEPS_PER_EPOCH),
    sampler=WholeBufferSampler(),
    backend=LocalBackend(),
)


# 4. Optimizer and Loss Pipeline
# TODO: add LR Annealing linearly to 0
optimizer = {"default": torch.optim.Adam(agent_network.parameters(), lr=LEARNING_RATE)}

pol_rep = agent_network.components["policy_head"].representation
val_rep = agent_network.components["value_head"].representation
loss_pipeline = LossPipeline(
    modules=[
        ClippedSurrogateLoss(
            device=DEVICE,
            representation=pol_rep,
            clip_param=CLIP_PARAM,
            entropy_coefficient=ENTROPY_COEF,  # TODO: ensure entropy coeff is used and entropy loss is done etc.
            optimizer_name="default",
        ),
        ClippedValueLoss(
            device=DEVICE,
            representation=val_rep,
            clip_param=CLIP_PARAM,
            target_key="returns",
            optimizer_name="default",
            loss_factor=VALUE_COEF,
        ),
    ],
    minibatch_size=STEPS_PER_EPOCH // NUM_MINIBATCHES,
    num_actions=num_actions,
    unroll_steps=0,  # PPO is single-step
)


target_builder = TargetBuilderPipeline(
    [
        PassThroughTargetBuilder(
            ["values", "returns", "actions", "old_log_probs", "advantages"]
        ),
        SingleStepFormatter(),
    ]
)


# 5. Learner
learner = UniversalLearner(
    agent_network=agent_network,
    device=DEVICE,
    num_actions=num_actions,
    observation_dimensions=obs_dim,
    observation_dtype=torch.float32,
    loss_pipeline=loss_pipeline,
    optimizer=optimizer,
    lr_scheduler={},
    target_builder=target_builder,
    callbacks=[MetricEarlyStopCallback(threshold=TARGET_KL)],
    clipnorm=0.5,  # TODO: ensure this actually works
)

print("All components initialized successfully!")

All components initialized successfully!


In [4]:
print("Starting explicit PPO training loop...")
start_time = time.time()

steps_collected = 0
current_episode_score = 0.0
state, info = env.reset()

while steps_collected < TOTAL_STEPS:
    # TODO: make this vectorized (use puffer)
    # TODO: add proper value bootstrapping and trunction for vectorized envs
    # ==========================================
    # PHASE 1: Trajectory Collection
    # ==========================================
    epoch_steps = 0
    trajectory_start_index = replay_buffer.size

    while epoch_steps < STEPS_PER_EPOCH:
        with torch.no_grad():
            obs_tensor = torch.tensor(
                state, dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)

            # 1. Inference
            result = policy_source.get_inference(obs=obs_tensor, info=info)
            action, metadata = action_selector.select_action(
                result=result,
                info=info,
                exploration=True,
            )

            log_prob = metadata["log_prob"]
            value = metadata["value"]
            action_val = action.item()

            # 2. Step Env
            next_state, reward, terminated, truncated, next_info = env.step(action_val)
            done = terminated or truncated

            # 3. Store Transition
            replay_buffer.store(
                observations=state,
                actions=action_val,
                values=float(value.item() if torch.is_tensor(value) else value),
                old_log_probs=float(
                    log_prob.item() if torch.is_tensor(log_prob) else log_prob
                ),
                rewards=reward,
                dones=done,
                info=info,
            )

            state = next_state
            info = next_info
            current_episode_score += reward
            epoch_steps += 1
            steps_collected += 1

            # 4. Handle Episode End / Bootstrap Value
            if done or epoch_steps == STEPS_PER_EPOCH:
                if terminated:
                    last_value = 0.0
                else:
                    # Bootstrap if truncated or epoch ended mid-episode
                    obs_t = torch.tensor(
                        state, dtype=torch.float32, device=DEVICE
                    ).unsqueeze(0)
                    out = agent_network.obs_inference(obs_t)
                    last_value = out.value.item()

                # Process GAE for the finished trajectory slice
                trajectory_end_index = replay_buffer.size
                trajectory_slice = slice(trajectory_start_index, trajectory_end_index)

                if trajectory_end_index > trajectory_start_index:
                    res = replay_buffer.input_processor.finish_trajectory(
                        replay_buffer.buffers,
                        trajectory_slice,
                        last_value=last_value,
                    )
                    if res:
                        for k, v in res.items():
                            replay_buffer.buffers[k][trajectory_slice] = v

                trajectory_start_index = trajectory_end_index

                if done:
                    print(
                        f"Step {steps_collected} | Episode Score: {current_episode_score}"
                    )
                    state, info = env.reset()
                    current_episode_score = 0.0

    # ==========================================
    # PHASE 2: Optimization (Learning)
    # ==========================================
    iterator = PPOEpochIterator(
        replay_buffer=replay_buffer,
        num_epochs=TRAIN_POLICY_ITERATIONS,
        num_minibatches=NUM_MINIBATCHES,
        device=DEVICE,
    )

    for step_metrics in learner.step(iterator):
        # Optional: Print out step_metrics here to monitor loss
        pass

    # Clear buffer for next epoch
    replay_buffer.clear()

total_time = time.time() - start_time
print(f"Training finished in {total_time:.2f} seconds.")

# TODO: ensure correct debug variables:
# policy_loss: the mean policy loss across all data points.
# value_loss: the mean value loss across all data points.
# entropy_loss: the mean entropy value across all data points.
# clipfrac: the fraction of the training data that triggered the clipped objective.
# approxkl: the approximate Kullback–Leibler divergence, measured by (-logratio).mean(), which corresponds to the k1 estimator in John Schulman’s blog post on approximating KL divergence. This blog post also suggests using an alternative estimator ((ratio - 1) - logratio).mean(), which is unbiased and has less variance.

Starting explicit PPO training loop...
Step 16 | Episode Score: 16.0
Step 27 | Episode Score: 11.0
Step 41 | Episode Score: 14.0
Step 63 | Episode Score: 22.0
Step 74 | Episode Score: 11.0
Step 90 | Episode Score: 16.0
Step 115 | Episode Score: 25.0
Step 137 | Episode Score: 22.0
Step 163 | Episode Score: 26.0
Step 182 | Episode Score: 19.0
Step 202 | Episode Score: 20.0
Step 222 | Episode Score: 20.0
Step 239 | Episode Score: 17.0
Step 267 | Episode Score: 28.0
Step 324 | Episode Score: 57.0
Step 344 | Episode Score: 20.0
Step 356 | Episode Score: 12.0
Step 377 | Episode Score: 21.0
Step 415 | Episode Score: 38.0
Step 427 | Episode Score: 12.0
Step 443 | Episode Score: 16.0
Step 465 | Episode Score: 22.0
Step 476 | Episode Score: 11.0
Step 493 | Episode Score: 17.0
Step 510 | Episode Score: 17.0
Step 536 | Episode Score: 26.0
Step 547 | Episode Score: 11.0
Step 561 | Episode Score: 14.0
Step 595 | Episode Score: 34.0
Step 611 | Episode Score: 16.0
Step 624 | Episode Score: 13.0
Step 6